#### Transform Orders Data - Explode Arrays
1. Access elements from JSON object
2. Deduplicate Array Elements
3. Exlpode Arrays
4. Write transformed data to silver schema

In [0]:
Select * from gizmobox_catalog_subbu.silver.orders_json

##### 1. Access elements from JSON object

In [0]:
select json_value.order_id,
       json_value.total_amount,
       json_value.order_status,
       json_value.payment_method,
       json_value.customer_id,
       json_value.transaction_timestamp,
       json_value.items
 from gizmobox_catalog_subbu.silver.orders_json

##### 2. Deduplicate Array Elements

In [0]:
select json_value.order_id,
       json_value.total_amount,
       json_value.order_status,
       json_value.payment_method,
       json_value.customer_id,
       json_value.transaction_timestamp,
      array_distinct(json_value.items) as Items
 from gizmobox_catalog_subbu.silver.orders_json

##### 3. Exlpode Arrays

In [0]:
CREATE OR REPLACE TEMP VIEW tv_orders_explode as
select json_value.order_id,
       json_value.total_amount,
       json_value.order_status,
       json_value.payment_method,
       json_value.customer_id,
       json_value.transaction_timestamp,
      explode(array_distinct(json_value.items)) as Item
 from gizmobox_catalog_subbu.silver.orders_json

In [0]:
SELECT order_id,
       order_status,
       payment_method,
       total_amount,       
       transaction_timestamp,
       customer_id,
       Item.item_id,
       Item.name,
       Item.price,
       Item.quantity,
       Item.category,
       Item.details.brand,
       Item.details.color
FROM 
    tv_orders_explode

##### 4. Write transformed data to silver schema

In [0]:
CREATE TABLE gizmobox_catalog_subbu.silver.orders
SELECT order_id,
       order_status,
       payment_method,
       total_amount,       
       transaction_timestamp,
       customer_id,
       Item.item_id,
       Item.name,
       Item.price,
       Item.quantity,
       Item.category,
       Item.details.brand,
       Item.details.color
FROM 
    tv_orders_explode

In [0]:
select * from gizmobox_catalog_subbu.silver.orders